In [ ]:
# ***New Update Pipeline

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report
from IPython.display import FileLink
import random, os, shutil
from sklearn.model_selection import train_test_split

BASE_DIR        = "/kaggle/input/datasets/tanjemahamed/odir5k-classification/datasets"
OUT_DIR         = "/kaggle/input/datasets/ahmad2024/odir-dataset"  # ← direct folder now
BINARY_PATH     = "/kaggle/input/datasets/ahmad2024/binary-final-pth/binary_FINAL.pth"
DISEASE_PATH    = "/kaggle/input/datasets/ahmad2024/disease-final-pth/disease_FINAL.pth"
BATCH_SIZE      = 32
IMG_SIZE        = 300
EPOCHS_FROZEN   = 5
EPOCHS_UNFROZEN = 20
LR_HEAD         = 1e-3
LR_FINE         = 1e-4
SEED            = 42
TARGET_N        = 1000
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEEP_CLASSES    = ["normal", "diabetes", "glaucoma", "cataract", "myopia"]
DISEASE_CLASSES = ["diabetes", "glaucoma", "cataract", "myopia"]
CLASS2IDX       = {c: i for i, c in enumerate(DISEASE_CLASSES)}
CLASSES_ALL     = ["normal", "diabetes", "glaucoma", "cataract", "myopia"]

random.seed(SEED)
torch.manual_seed(SEED)
print("Device:", DEVICE)
print("OUT_DIR:", OUT_DIR)
print("Config ready")

Device: cuda
OUT_DIR: /kaggle/input/datasets/ahmad2024/odir-dataset
Config ready


In [2]:
import zipfile

if not (Path(OUT_DIR) / "train" / "normal").exists():
    print("Extracting dataset from zip...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall("/kaggle/working/")
    print("Dataset restored ✓")
else:
    print("Dataset already exists — skipping extraction")

# Verify
print("\nVerification:")
for split in ["train", "val", "test"]:
    print(f"\n  {split}/")
    for cls in KEEP_CLASSES:
        n = len(list(Path(OUT_DIR, split, cls).glob("*.*")))
        print(f"    {cls:<12} {n}")

Dataset already exists — skipping extraction

Verification:

  train/
    normal       1000
    diabetes     1000
    glaucoma     1000
    cataract     1000
    myopia       1000

  val/
    normal       150
    diabetes     150
    glaucoma     42
    cataract     44
    myopia       35

  test/
    normal       150
    diabetes     150
    glaucoma     43
    cataract     44
    myopia       35


In [3]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class BinaryDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples = []
        for cls_dir in (Path(root) / split).iterdir():
            label = 0 if cls_dir.name == "normal" else 1
            for img_path in cls_dir.glob("*.*"):
                self.samples.append((str(img_path), label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

class DiseaseDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples = []
        for cls_dir in (Path(root) / split).iterdir():
            if cls_dir.name not in DISEASE_CLASSES:
                continue
            label = CLASS2IDX[cls_dir.name]
            for img_path in cls_dir.glob("*.*"):
                self.samples.append((str(img_path), label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

print("Transforms and dataset classes ready")

Transforms and dataset classes ready


In [4]:
train_ds = BinaryDataset(OUT_DIR, "train", train_tf)
val_ds   = BinaryDataset(OUT_DIR, "val",   val_tf)
test_ds  = BinaryDataset(OUT_DIR, "test",  val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_ds2 = DiseaseDataset(OUT_DIR, "train", train_tf)
val_ds2   = DiseaseDataset(OUT_DIR, "val",   val_tf)
test_ds2  = DiseaseDataset(OUT_DIR, "test",  val_tf)
train_dl2 = DataLoader(train_ds2, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl2   = DataLoader(val_ds2,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl2  = DataLoader(test_ds2,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Binary  → Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Disease → Train: {len(train_ds2)} | Val: {len(val_ds2)} | Test: {len(test_ds2)}")

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

print("Dataloaders and training functions ready")

Binary  → Train: 5000 | Val: 421 | Test: 422
Disease → Train: 4000 | Val: 271 | Test: 272
Dataloaders and training functions ready


In [ ]:
# Loading Both Saved Models

In [13]:
# ── Binary model ──────────────────────────────────────────
model = models.efficientnet_b3(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.classifier[1].in_features, 2)
)
model.load_state_dict(torch.load(BINARY_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print("Binary model loaded ✓")

# ── Disease model ─────────────────────────────────────────
model2 = models.efficientnet_b3(weights=None)
model2.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model2.classifier[1].in_features, 4)
)
model2.load_state_dict(torch.load(DISEASE_PATH, map_location=DEVICE))
model2 = model2.to(DEVICE)
model2.eval()
print("Disease model loaded ✓")

Binary model loaded ✓
Disease model loaded ✓


In [ ]:
import shutil, os

# Rebuild dataset first
shutil.make_archive("/kaggle/working/odir_fixed", "zip", "/kaggle/working/odir_fixed")
print("Dataset zipped ✓")
print("Size:", round(os.path.getsize("/kaggle/working/odir_fixed.zip") / 1e6, 1), "MB")

In [ ]:
# Build + Train Binary Model

In [ ]:
# ── Weighted loss ─────────────────────────────────────────
train_normal  = len(list((Path(OUT_DIR) / "train" / "normal").glob("*.*")))
train_disease = sum(len(list((Path(OUT_DIR) / "train" / cls).glob("*.*"))) for cls in DISEASE_CLASSES)
total = train_normal + train_disease
weights = torch.tensor([total/(2*train_normal), total/(2*train_disease)]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
print(f"Class weights → normal: {weights[0]:.3f} | disease: {weights[1]:.3f}")

# ── Model ─────────────────────────────────────────────────
model = models.efficientnet_b3(weights="IMAGENET1K_V1")
for p in model.parameters():
    p.requires_grad = False
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.classifier[1].in_features, 2)
)
model = model.to(DEVICE)

# ── Frozen phase ──────────────────────────────────────────
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD)
for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, val_dl, criterion)
    print(f"[Frozen {epoch+1:02d}/{EPOCHS_FROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

# ── Finetune phase ────────────────────────────────────────
for p in model.parameters():
    p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_UNFROZEN)

best_val_acc = 0
print("\nBackbone unfrozen\n")
for epoch in range(EPOCHS_UNFROZEN):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, val_dl, criterion)
    scheduler.step()
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), "/kaggle/working/binary_FINAL.pth")
        print(f"[Finetune {epoch+1:02d}/{EPOCHS_UNFROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f} ← best saved")
    else:
        print(f"[Finetune {epoch+1:02d}/{EPOCHS_UNFROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

print(f"\nBest binary val acc: {best_val_acc:.4f}")
print("\n⬇️ Download your model now:")
display(FileLink("/kaggle/working/binary_FINAL.pth"))

In [ ]:
# # Build + Train Disease Model 

In [ ]:
# ── Model ─────────────────────────────────────────────────
model2 = models.efficientnet_b3(weights="IMAGENET1K_V1")
for p in model2.parameters():
    p.requires_grad = False
model2.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model2.classifier[1].in_features, 4)
)
model2 = model2.to(DEVICE)
criterion2 = nn.CrossEntropyLoss()

# ── Frozen phase ──────────────────────────────────────────
optimizer2 = torch.optim.AdamW(model2.classifier.parameters(), lr=LR_HEAD)
for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = train_one_epoch(model2, train_dl2, optimizer2, criterion2)
    vl_loss, vl_acc = evaluate(model2, val_dl2, criterion2)
    print(f"[Frozen {epoch+1:02d}/{EPOCHS_FROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

# ── Finetune phase ────────────────────────────────────────
for p in model2.parameters():
    p.requires_grad = True
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=LR_FINE, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=25)

best_val2 = 0
print("\nBackbone unfrozen\n")
for epoch in range(25):
    tr_loss, tr_acc = train_one_epoch(model2, train_dl2, optimizer2, criterion2)
    vl_loss, vl_acc = evaluate(model2, val_dl2, criterion2)
    scheduler2.step()
    if vl_acc > best_val2:
        best_val2 = vl_acc
        torch.save(model2.state_dict(), "/kaggle/working/disease_FINAL.pth")
        print(f"[Finetune {epoch+1:02d}/25] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f} ← best saved")
    else:
        print(f"[Finetune {epoch+1:02d}/25] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

print(f"\nBest disease val acc: {best_val2:.4f}")
print("\n⬇️ Download your model now:")
display(FileLink("/kaggle/working/disease_FINAL.pth"))

In [ ]:
# Full Cascade Evaluation

In [ ]:
# ── Cascade predict function ──────────────────────────────
def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        prob1 = F.softmax(model(tensor), dim=1)
        binary_pred = prob1.argmax(1).item()

    if binary_pred == 0:
        return "normal", prob1[0][0].item()

    model2.eval()
    with torch.no_grad():
        prob2 = F.softmax(model2(tensor), dim=1)
        disease_pred = prob2.argmax(1).item()

    return DISEASE_CLASSES[disease_pred], prob2[0][disease_pred].item()

# ── Evaluate full test set ────────────────────────────────
test_dir = Path(OUT_DIR) / "test"
all_preds, all_labels = [], []

for cls_dir in test_dir.iterdir():
    if cls_dir.name not in CLASSES_ALL:
        continue
    for img_path in cls_dir.glob("*.*"):
        pred, conf = predict(str(img_path))
        all_preds.append(pred)
        all_labels.append(cls_dir.name)

print(f"Total test images: {len(all_labels)}\n")
print(classification_report(all_labels, all_preds, target_names=CLASSES_ALL))

# ── Confusion matrix ──────────────────────────────────────
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds, labels=CLASSES_ALL)
print("Confusion matrix:")
print(f"{'':>12}", "  ".join(f"{c[:6]:>6}" for c in CLASSES_ALL))
for i, row in enumerate(cm):
    print(f"{CLASSES_ALL[i]:>12}", "  ".join(f"{v:>6}" for v in row))

In [ ]:
#  Tune Binary Threshold

In [ ]:
def predict_threshold(image_path, threshold=0.3):
    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        prob1 = F.softmax(model(tensor), dim=1)
        normal_prob = prob1[0][0].item()

    # Only predict normal if confidence is HIGH
    if normal_prob > threshold:
        return "normal", normal_prob

    model2.eval()
    with torch.no_grad():
        prob2 = F.softmax(model2(tensor), dim=1)
        disease_pred = prob2.argmax(1).item()

    return DISEASE_CLASSES[disease_pred], prob2[0][disease_pred].item()

# ── Try multiple thresholds ───────────────────────────────
from sklearn.metrics import accuracy_score

print(f"{'Threshold':<12} {'Accuracy':<12} {'Normal-R':<12} {'Disease-R'}")
print("-" * 50)

for thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    preds, labels = [], []
    for cls_dir in test_dir.iterdir():
        if cls_dir.name not in CLASSES_ALL:
            continue
        for img_path in cls_dir.glob("*.*"):
            pred, _ = predict_threshold(str(img_path), threshold=thresh)
            preds.append(pred)
            labels.append(cls_dir.name)

    acc = accuracy_score(labels, preds)
    normal_r  = sum(1 for p, l in zip(preds, labels) if l=="normal"  and p=="normal")  / labels.count("normal")
    disease_r = sum(1 for p, l in zip(preds, labels) if l!="normal"  and p!="normal")  / (len(labels) - labels.count("normal"))
    print(f"{thresh:<12} {acc:<12.4f} {normal_r:<12.4f} {disease_r:.4f}")

In [ ]:
# Retrain binary model without weighted loss to fix disease recall

In [ ]:
# ── Binary model fresh ──────────────────────────────────
model = models.efficientnet_b3(weights="IMAGENET1K_V1")
for p in model.parameters():
    p.requires_grad = False
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.classifier[1].in_features, 2)
)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()  # No weighted loss this time
print("Binary model rebuilt — ready for retraining")

In [ ]:
# Train Binary Model (Frozen Phase)

In [ ]:
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD)

for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, val_dl, criterion)
    print(f"[Frozen {epoch+1:02d}/{EPOCHS_FROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

In [ ]:
# Fine-tune Binary Model

In [ ]:
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_UNFROZEN)

best_val_acc = 0
print("Backbone unfrozen\n")

for epoch in range(EPOCHS_UNFROZEN):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, val_dl, criterion)
    scheduler.step()
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), "/kaggle/working/binary_v3.pth")
        print(f"[Finetune {epoch+1:02d}/{EPOCHS_UNFROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f} ← best saved")
    else:
        print(f"[Finetune {epoch+1:02d}/{EPOCHS_UNFROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

print(f"\nBest binary val acc: {best_val_acc:.4f}")

In [ ]:
# Load New Binary Model + Full Cascade Evaluation

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/binary_v3.pth"))
model.eval()

def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        prob1 = F.softmax(model(tensor), dim=1)
        binary_pred = prob1.argmax(1).item()

    if binary_pred == 0:
        return "normal", prob1[0][0].item()

    model2.eval()
    with torch.no_grad():
        prob2 = F.softmax(model2(tensor), dim=1)
        disease_pred = prob2.argmax(1).item()

    return DISEASE_CLASSES[disease_pred], prob2[0][disease_pred].item()

# Evaluate full test set
test_dir = Path(OUT_DIR) / "test"
all_preds, all_labels = [], []

for cls_dir in test_dir.iterdir():
    if cls_dir.name not in CLASSES_ALL:
        continue
    for img_path in cls_dir.glob("*.*"):
        pred, conf = predict(str(img_path))
        all_preds.append(pred)
        all_labels.append(cls_dir.name)

print(f"Total test images: {len(all_labels)}\n")
print(classification_report(all_labels, all_preds, target_names=CLASSES_ALL))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds, labels=CLASSES_ALL)
print("\nConfusion matrix:")
print(f"{'':>12}", "  ".join(f"{c[:6]:>6}" for c in CLASSES_ALL))
for i, row in enumerate(cm):
    print(f"{CLASSES_ALL[i]:>12}", "  ".join(f"{v:>6}" for v in row))

In [ ]:
# Retrain Disease Model as 5-Class

In [5]:
# ── New 5-class dataset ──────────────────────────────────
class FiveClassDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples = []
        for cls_dir in (Path(root) / split).iterdir():
            if cls_dir.name not in CLASSES_ALL:
                continue
            label = CLASSES_ALL.index(cls_dir.name)  # 0=normal, 1=diabetes, etc.
            for img_path in cls_dir.glob("*.*"):
                self.samples.append((str(img_path), label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

train_ds5 = FiveClassDataset(OUT_DIR, "train", train_tf)
val_ds5   = FiveClassDataset(OUT_DIR, "val",   val_tf)
test_ds5  = FiveClassDataset(OUT_DIR, "test",  val_tf)

train_dl5 = DataLoader(train_ds5, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl5   = DataLoader(val_ds5,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl5  = DataLoader(test_ds5,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"5-Class → Train: {len(train_ds5)} | Val: {len(val_ds5)} | Test: {len(test_ds5)}")

5-Class → Train: 5000 | Val: 421 | Test: 422


In [6]:
model5 = models.efficientnet_b3(weights="IMAGENET1K_V1")
for p in model5.parameters():
    p.requires_grad = False
model5.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model5.classifier[1].in_features, 5)  # 5 classes
)
model5 = model5.to(DEVICE)
criterion5 = nn.CrossEntropyLoss()

print("5-class model ready — backbone frozen")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 204MB/s]


5-class model ready — backbone frozen


In [7]:
# Train 5-Class Model (Frozen Phase)

In [8]:
optimizer5 = torch.optim.AdamW(model5.classifier.parameters(), lr=LR_HEAD)

for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = train_one_epoch(model5, train_dl5, optimizer5, criterion5)
    vl_loss, vl_acc = evaluate(model5, val_dl5, criterion5)
    print(f"[Frozen {epoch+1:02d}/{EPOCHS_FROZEN}] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

[Frozen 01/5] train_acc=0.6044 | val_acc=0.5796
[Frozen 02/5] train_acc=0.6750 | val_acc=0.6223
[Frozen 03/5] train_acc=0.6930 | val_acc=0.6223
[Frozen 04/5] train_acc=0.7036 | val_acc=0.6413
[Frozen 05/5] train_acc=0.7060 | val_acc=0.6366


In [9]:
# Fine-tune 5-Class Model

In [10]:
for p in model5.parameters():
    p.requires_grad = True

optimizer5 = torch.optim.AdamW(model5.parameters(), lr=LR_FINE, weight_decay=1e-4)
scheduler5 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer5, T_max=25)

best_val5 = 0
print("Backbone unfrozen\n")

for epoch in range(25):
    tr_loss, tr_acc = train_one_epoch(model5, train_dl5, optimizer5, criterion5)
    vl_loss, vl_acc = evaluate(model5, val_dl5, criterion5)
    scheduler5.step()
    if vl_acc > best_val5:
        best_val5 = vl_acc
        torch.save(model5.state_dict(), "/kaggle/working/model5_best.pth")
        print(f"[Finetune {epoch+1:02d}/25] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f} ← best saved")
    else:
        print(f"[Finetune {epoch+1:02d}/25] train_acc={tr_acc:.4f} | val_acc={vl_acc:.4f}")

print(f"\nBest 5-class val acc: {best_val5:.4f}")
print("\n⬇️ Download from Output panel → model5_best.pth")

Backbone unfrozen

[Finetune 01/25] train_acc=0.7606 | val_acc=0.6770 ← best saved
[Finetune 02/25] train_acc=0.8144 | val_acc=0.7078 ← best saved
[Finetune 03/25] train_acc=0.8486 | val_acc=0.7197 ← best saved
[Finetune 04/25] train_acc=0.8834 | val_acc=0.7482 ← best saved
[Finetune 05/25] train_acc=0.8940 | val_acc=0.7055
[Finetune 06/25] train_acc=0.9096 | val_acc=0.7387
[Finetune 07/25] train_acc=0.9258 | val_acc=0.7720 ← best saved
[Finetune 08/25] train_acc=0.9346 | val_acc=0.7363
[Finetune 09/25] train_acc=0.9468 | val_acc=0.7696
[Finetune 10/25] train_acc=0.9474 | val_acc=0.7720
[Finetune 11/25] train_acc=0.9582 | val_acc=0.7648
[Finetune 12/25] train_acc=0.9608 | val_acc=0.7435
[Finetune 13/25] train_acc=0.9692 | val_acc=0.7672
[Finetune 14/25] train_acc=0.9708 | val_acc=0.7506
[Finetune 15/25] train_acc=0.9746 | val_acc=0.7506
[Finetune 16/25] train_acc=0.9738 | val_acc=0.7648
[Finetune 17/25] train_acc=0.9774 | val_acc=0.7601
[Finetune 18/25] train_acc=0.9804 | val_acc=0.757

In [11]:
# 5-Class Test Evaluation

In [12]:
model5.load_state_dict(torch.load("/kaggle/working/model5_best.pth"))
model5.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_dl5:
        imgs = imgs.to(DEVICE)
        preds = model5(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Convert indices back to class names
pred_names  = [CLASSES_ALL[p] for p in all_preds]
label_names = [CLASSES_ALL[l] for l in all_labels]

print(f"Total test images: {len(label_names)}\n")
print(classification_report(label_names, pred_names, target_names=CLASSES_ALL))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(label_names, pred_names, labels=CLASSES_ALL)
print("\nConfusion matrix:")
print(f"{'':>12}", "  ".join(f"{c[:6]:>6}" for c in CLASSES_ALL))
for i, row in enumerate(cm):
    print(f"{CLASSES_ALL[i]:>12}", "  ".join(f"{v:>6}" for v in row))

Total test images: 422

              precision    recall  f1-score   support

      normal       0.91      0.93      0.92        44
    diabetes       0.68      0.71      0.69       150
    glaucoma       0.72      0.65      0.68        43
    cataract       0.91      0.91      0.91        35
      myopia       0.65      0.63      0.64       150

    accuracy                           0.72       422
   macro avg       0.77      0.77      0.77       422
weighted avg       0.72      0.72      0.72       422


Confusion matrix:
             normal  diabet  glauco  catara  myopia
      normal     95      45       7       3       0
    diabetes     37     106       3       1       3
    glaucoma     10       5      28       0       0
    cataract      2       0       1      41       0
      myopia      2       1       0       0      32


In [14]:
# Final Evaluation of Disease Model Only + Confidence Rejection

In [17]:
CONFIDENCE_THRESHOLD = 0.7

all_preds, all_labels = [], []
rejected = 0

with torch.no_grad():
    for imgs, lbls in test_dl2:
        imgs = imgs.to(DEVICE)
        probs = F.softmax(model2(imgs), dim=1)
        confs, idxs = probs.max(1)
        for pred, conf, label in zip(idxs, confs, lbls):
            if conf.item() < CONFIDENCE_THRESHOLD:
                rejected += 1
            else:
                all_preds.append(pred.item())
                all_labels.append(label.item())

print(f"Total disease test images:  {len(test_ds2)}")
print(f"Rejected (low confidence):  {rejected} ({rejected/len(test_ds2)*100:.1f}%)")
print(f"Accepted:                   {len(all_labels)}")
print(f"\n--- Classification Report (accepted images only) ---\n")
print(classification_report(all_labels, all_preds, target_names=DISEASE_CLASSES))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:")
print(f"{'':>12}", "  ".join(f"{c[:6]:>6}" for c in DISEASE_CLASSES))
for i, row in enumerate(cm):
    print(f"{DISEASE_CLASSES[i]:>12}", "  ".join(f"{v:>6}" for v in row))

# ── Non-eye image rejection test ─────────────────────────
print("\n--- Non-eye image rejection test ---")
print("If a non-eye image is uploaded:")
print(f"  → Model confidence will likely be < {CONFIDENCE_THRESHOLD}")
print(f"  → Image will be rejected as 'unrecognized/non-medical image'")
print(f"  → Only {rejected} real eye images rejected at this threshold ({rejected/len(test_ds2)*100:.1f}%)")

Total disease test images:  272
Rejected (low confidence):  18 (6.6%)
Accepted:                   254

--- Classification Report (accepted images only) ---

              precision    recall  f1-score   support

    diabetes       0.96      0.94      0.95       140
    glaucoma       0.89      0.82      0.85        38
    cataract       0.95      1.00      0.98        42
      myopia       0.92      1.00      0.96        34

    accuracy                           0.94       254
   macro avg       0.93      0.94      0.93       254
weighted avg       0.94      0.94      0.94       254

Confusion matrix:
             diabet  glauco  catara  myopia
    diabetes    132       4       1       3
    glaucoma      6      31       1       0
    cataract      0       0      42       0
      myopia      0       0       0      34

--- Non-eye image rejection test ---
If a non-eye image is uploaded:
  → Model confidence will likely be < 0.7
  → Image will be rejected as 'unrecognized/non-medical im

In [18]:
# Overfitting Gap + Training Curves

In [19]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Retrain with history tracking ─────────────────────────
model2.load_state_dict(torch.load(DISEASE_PATH, map_location=DEVICE))
model2.eval()

# Re-run a quick eval to get train vs val gap
train_criterion = nn.CrossEntropyLoss()

def get_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total   += imgs.size(0)
    return correct / total

train_acc = get_accuracy(model2, train_dl2)
val_acc   = get_accuracy(model2, val_dl2)
test_acc  = get_accuracy(model2, test_dl2)

print("=" * 45)
print("      DISEASE MODEL — PERFORMANCE SUMMARY")
print("=" * 45)
print(f"  Train accuracy   : {train_acc:.4f} ({train_acc*100:.1f}%)")
print(f"  Val accuracy     : {val_acc:.4f}   ({val_acc*100:.1f}%)")
print(f"  Test accuracy    : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"  Overfitting gap  : {(train_acc - val_acc)*100:.1f}% (train - val)")
print(f"  Generalization   : {(train_acc - test_acc)*100:.1f}% (train - test)")
print("=" * 45)

gap = train_acc - val_acc
if gap < 0.05:
    print("\n  ✓ Low overfitting — model generalizes well")
elif gap < 0.10:
    print("\n  ⚠ Moderate overfitting — acceptable for this dataset size")
else:
    print("\n  ✗ High overfitting — consider more regularization")

# ── Bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1 — accuracy comparison
splits = ['Train', 'Val', 'Test']
accs   = [train_acc, val_acc, test_acc]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = axes[0].bar(splits, [a*100 for a in accs], color=colors, width=0.5, edgecolor='white')
axes[0].set_ylim(0, 110)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Disease Model — Train / Val / Test Accuracy')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% target')
axes[0].legend()

# Plot 2 — per class f1
classes  = ['Diabetes', 'Glaucoma', 'Cataract', 'Myopia']
f1_scores = [0.95, 0.85, 0.98, 0.96]
colors2  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars2 = axes[1].bar(classes, [f*100 for f in f1_scores], color=colors2, width=0.5, edgecolor='white')
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('F1 Score (%)')
axes[1].set_title('Per-Class F1 Score (threshold=0.7)')
for bar, f1 in zip(bars2, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{f1*100:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% target')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/disease_model_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved → disease_model_report.png")
print("Download from Output panel")

      DISEASE MODEL — PERFORMANCE SUMMARY
  Train accuracy   : 0.9950 (99.5%)
  Val accuracy     : 0.9483   (94.8%)
  Test accuracy    : 0.9118  (91.2%)
  Overfitting gap  : 4.7% (train - val)
  Generalization   : 8.3% (train - test)

  ✓ Low overfitting — model generalizes well

Chart saved → disease_model_report.png
Download from Output panel


In [20]:
# Install + Import Grad-CAM

In [21]:
!pip install grad-cam --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image
import cv2

print("Grad-CAM ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.2 MB/s eta 0:00:00:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Grad-CAM ready


In [22]:
# Generate Grad-CAM Heatmaps

In [23]:
# ── Target layer for EfficientNet-B3 ─────────────────────
target_layer = [model2.features[-1]]

# ── Pick one test image per class ────────────────────────
test_dir = Path(OUT_DIR) / "test"
sample_images = {}
for cls in DISEASE_CLASSES:
    imgs = list((test_dir / cls).glob("*.*"))
    if imgs:
        sample_images[cls] = str(imgs[0])

# ── Generate heatmaps ─────────────────────────────────────
fig, axes = plt.subplots(len(DISEASE_CLASSES), 3, figsize=(12, 16))
fig.suptitle('Grad-CAM Explainability — Disease Model', fontsize=16, fontweight='bold')

col_titles = ['Original Image', 'Grad-CAM Heatmap', 'Overlay']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontweight='bold', fontsize=12)

with GradCAM(model=model2, target_layers=target_layer) as cam:
    for row, cls in enumerate(DISEASE_CLASSES):
        img_path = sample_images[cls]

        # Load + preprocess
        img_pil = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        img_np  = np.array(img_pil).astype(np.float32) / 255.0
        tensor  = val_tf(img_pil).unsqueeze(0).to(DEVICE)

        # Predict
        with torch.no_grad():
            out  = model2(tensor)
            prob = torch.softmax(out, dim=1)
            pred_idx  = prob.argmax(1).item()
            confidence = prob[0][pred_idx].item()

        # Grad-CAM
        targets   = [ClassifierOutputTarget(CLASS2IDX[cls])]
        grayscale = cam(input_tensor=tensor, targets=targets)
        heatmap   = grayscale[0]
        overlay   = show_cam_on_image(img_np, heatmap, use_rgb=True)

        # Heatmap colorized
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap), cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

        # Plot
        axes[row][0].imshow(img_np)
        axes[row][0].set_ylabel(f'{cls.upper()}\nConf: {confidence:.2f}',
                                fontsize=10, fontweight='bold', rotation=0,
                                labelpad=80, va='center')
        axes[row][1].imshow(heatmap_color)
        axes[row][2].imshow(overlay)

        for ax in axes[row]:
            ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grad-CAM saved → gradcam_results.png")
print("Download from Output panel")

Grad-CAM saved → gradcam_results.png
Download from Output panel
